# Visualizations for the AI4AM abstract

MatterGen metastable SMACT-valid novelty summary and one substituted-match example.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatviz import structure_2d
from tabulate import tabulate

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_utils import find_repo_root  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
for import_path in (ROOT, NOTEBOOKS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from notebook_constants import (  # noqa: E402
    CRYSTAL_SYSTEM_ORDER,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    load_direct_matches,
    load_pickle_gz,
    missing_required_paths,
    required_paths,
)
from plot_style import (  # noqa: E402
    CATEGORY_COLORS,
    GRAY,
    WHITE,
    apply_plot_style,
)

from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402

MODEL = "mattergen"
SIMPLE_CATEGORY_ORDER = ["1", "2", "3"]
SIMPLE_CATEGORY_LABELS = {
    "1": "Exact match",
    "2": "Substituted match",
    "3": "No match",
}
SIMPLE_CATEGORY_COLORS = {
    "1": CATEGORY_COLORS["1"],
    "2": CATEGORY_COLORS["2-1"],
    "3": CATEGORY_COLORS["3"],
}

EXAMPLE_ENTRY_IDX = 20297
EXAMPLE_GEN_IDX = 8035
EXAMPLE_TRAIN_IDX = 3438

ROOT, INPUT_DIR, RESULTS_DIR

In [ ]:
AI4AM_PATH_KEYS = (
    "generated_structures",
    "training_structures",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
    "wyckoff_repr",
)

paths = required_paths(MODEL, INPUT_DIR, RESULTS_DIR, AI4AM_PATH_KEYS)
missing_paths = pd.DataFrame(missing_required_paths(paths))
if not missing_paths.empty:
    display(missing_paths)
    raise FileNotFoundError(
        f"Missing required files for MODEL={MODEL!r}. Check INPUT_DIR and RESULTS_DIR."
    )

paths

## Metastable SMACT-valid category ratios

Category ratios by crystal system for MatterGen samples with relaxed hull energy <= 0.1 eV/atom and SMACT-valid composition.

In [ ]:
classifications = classify_model(
    MODEL,
    paths,
    include_crystal_system=True,
    include_simple_category=True,
    simple_category_labels=SIMPLE_CATEGORY_LABELS,
    simple_category_order=SIMPLE_CATEGORY_ORDER,
)
selected_classifications = classifications[
    classifications["is_metastable_smact_valid"]
].copy()
classifications.head()

In [ ]:
apply_plot_style()


def build_crystal_system_category_ratios(
    selected: pd.DataFrame,
) -> pd.DataFrame:
    if selected.empty:
        return pd.DataFrame(
            columns=["crystal_system", "simple_category", "count", "ratio"]
        )

    counts = (
        selected.groupby(["crystal_system", "simple_category"], observed=False)
        .size()
        .rename("count")
    )
    full_index = pd.MultiIndex.from_product(
        [CRYSTAL_SYSTEM_ORDER, SIMPLE_CATEGORY_ORDER],
        names=["crystal_system", "simple_category"],
    )
    result = counts.reindex(full_index, fill_value=0).reset_index()
    totals = result.groupby("crystal_system", observed=False)["count"].transform("sum")
    result["ratio"] = np.where(totals > 0, result["count"] / totals, 0.0)
    result["simple_category_label"] = result["simple_category"].map(
        SIMPLE_CATEGORY_LABELS
    )
    return result


def build_tabulated_crystal_system_ratios(counts_by_system: pd.DataFrame) -> str:
    if counts_by_system.empty:
        return "No classifications available."

    count_table = (
        counts_by_system.pivot(
            index="crystal_system", columns="simple_category", values="count"
        )
        .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=SIMPLE_CATEGORY_ORDER)
        .fillna(0)
        .astype(int)
    )
    totals = count_table.sum(axis=1)
    count_table = count_table[totals > 0]
    totals = totals[totals > 0]
    ratio_table = count_table.div(totals, axis=0)

    display_table = pd.DataFrame(index=count_table.index)
    for category in SIMPLE_CATEGORY_ORDER:
        display_table[SIMPLE_CATEGORY_LABELS[category]] = [
            f"{count:,} ({ratio:.3f})"
            for count, ratio in zip(
                count_table[category], ratio_table[category], strict=True
            )
        ]
    display_table.insert(0, "crystal_system", display_table.index)
    display_table = display_table.reset_index(drop=True)
    return tabulate(display_table, headers="keys", tablefmt="github", showindex=False)


def plot_category_ratios_by_crystal_system(counts_by_system: pd.DataFrame) -> None:
    if counts_by_system.empty:
        display(Markdown("No classifications available."))
        return

    values = (
        counts_by_system.pivot(
            index="crystal_system", columns="simple_category", values="ratio"
        )
        .reindex(index=CRYSTAL_SYSTEM_ORDER, columns=SIMPLE_CATEGORY_ORDER)
        .fillna(0.0)
    )
    totals = (
        counts_by_system.groupby("crystal_system", observed=False)["count"]
        .sum()
        .reindex(CRYSTAL_SYSTEM_ORDER, fill_value=0)
    )
    systems_with_data = totals[totals > 0].index.tolist()
    if not systems_with_data:
        display(Markdown("No classifications available."))
        return

    values = values.loc[systems_with_data]
    fig, ax = plt.subplots(figsize=(8.0, 6.0))
    bottom = np.zeros(len(values), dtype=float)
    x = np.arange(len(values))
    for category in SIMPLE_CATEGORY_ORDER:
        heights = values[category].to_numpy(dtype=float)
        ax.bar(
            x,
            heights,
            bottom=bottom,
            color=SIMPLE_CATEGORY_COLORS[category],
            edgecolor=WHITE,
            linewidth=0.5,
            label=SIMPLE_CATEGORY_LABELS[category],
        )
        bottom += heights

    ax.set_title(
        r"MatterGen samples ($E_\mathrm{hull} \leq 0.1$ [eV/atom], SMACT-valid)",
        fontsize=20,
        pad=40,
    )
    ax.set_ylabel("Ratio", fontsize=20)
    ax.set_ylim(0, 1)
    ax.set_xticks(x, systems_with_data, rotation=30, ha="right")
    ax.tick_params(axis="both", labelsize=16)
    ax.grid(axis="y", color=GRAY, alpha=0.35, linewidth=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 0.98),
        frameon=False,
        fontsize=16,
        ncol=3,
    )
    fig.tight_layout()
    plot_dir = RESULTS_DIR / "ai4am"
    plot_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_dir / "bar_chart.pdf", bbox_inches="tight")
    plt.show()


crystal_system_category_ratios = build_crystal_system_category_ratios(
    selected_classifications
)
display(Markdown(build_tabulated_crystal_system_ratios(crystal_system_category_ratios)))
plot_category_ratios_by_crystal_system(crystal_system_category_ratios)

## Substituted-match example

A space-group 225 or 229 pair from the relaxed Wyckoff substituted-match records with volume ratio >= 1.5, `cost_uniform=1`, exactly two distinct elements in each structure, `cost_mod_petti <= 5`, and no exact StructureMatcher match in `sm_fit.npz`.

In [ ]:
generated_structures: list[Structure] = load_pickle_gz(paths["generated_structures"])
training_structures: list[Structure] = load_pickle_gz(paths["training_structures"])
relaxed_wyckoff_records = load_pickle_gz(paths["relaxed_wyckoff_matches"])
direct_matches = load_direct_matches(paths["direct_sm"])


example_records = [
    record
    for record in relaxed_wyckoff_records
    if int(record["entry_idx"]) == EXAMPLE_ENTRY_IDX
]
if len(example_records) != 1:
    raise ValueError(
        f"Expected one record for entry_idx={EXAMPLE_ENTRY_IDX}, "
        f"found {len(example_records)}"
    )
example_record = example_records[0]

assert bool(example_record["match"])
assert int(example_record["gen_idx"]) == EXAMPLE_GEN_IDX
assert int(example_record["train_idx"]) == EXAMPLE_TRAIN_IDX
assert np.isclose(float(example_record["cost_uniform"]), 1.0)
assert float(example_record["cost_mod_petti"]) <= 5.0

example_classification = classifications[
    classifications["gen_idx"] == EXAMPLE_GEN_IDX
].iloc[0]
assert bool(example_classification["is_smact_valid"])
assert bool(example_classification["is_metastable"])
assert not bool(example_classification["is_direct_sm_match"])
assert EXAMPLE_GEN_IDX not in set(direct_matches["gen_idx"])

generated_example = generated_structures[EXAMPLE_GEN_IDX]
training_example = training_structures[EXAMPLE_TRAIN_IDX]
generated_analyzer = SpacegroupAnalyzer(generated_example)
training_analyzer = SpacegroupAnalyzer(training_example)
generated_spg_num = generated_analyzer.get_space_group_number()
training_spg_num = training_analyzer.get_space_group_number()
assert generated_spg_num in (225, 229)
assert training_spg_num in (225, 229)
assert len(generated_example.composition.elements) == 2
assert len(training_example.composition.elements) == 2

volume_ratio = max(generated_example.volume, training_example.volume) / min(
    generated_example.volume, training_example.volume
)
volume_difference = abs(generated_example.volume - training_example.volume)
assert volume_ratio >= 1.5

example_summary = pd.DataFrame(
    [
        {
            "role": "generated",
            "index": EXAMPLE_GEN_IDX,
            "formula": generated_example.composition.reduced_formula,
            "space_group": (
                f"{generated_analyzer.get_space_group_symbol()} ({generated_spg_num})"
            ),
            "atoms": len(generated_example),
            "distinct_elements": len(generated_example.composition.elements),
            "volume": generated_example.volume,
        },
        {
            "role": "train",
            "index": EXAMPLE_TRAIN_IDX,
            "formula": training_example.composition.reduced_formula,
            "space_group": (
                f"{training_analyzer.get_space_group_symbol()} ({training_spg_num})"
            ),
            "atoms": len(training_example),
            "distinct_elements": len(training_example.composition.elements),
            "volume": training_example.volume,
        },
    ]
)

display(
    Markdown(
        f"**entry_idx:** `{EXAMPLE_ENTRY_IDX}`  \n"
        f"**cost_uniform:** `{float(example_record['cost_uniform']):.6g}`  \n"
        f"**cost_mod_petti:** `{float(example_record['cost_mod_petti']):.6g}`  \n"
        f"**volume ratio:** `{volume_ratio:.6g}`  \n"
        f"**volume difference:** `{volume_difference:.6g}`"
    )
)
display(example_summary)

In [ ]:
def structure_title(role: str, index_label: str, structure: Structure) -> str:
    analyzer = SpacegroupAnalyzer(structure)
    spg = f"{analyzer.get_space_group_symbol()} ({analyzer.get_space_group_number()})"
    return (
        f"{role}<br>{index_label}<br>{structure.composition.reduced_formula}<br>{spg}"
    )


example_structures = {
    "Generated": generated_example,
    "Train": training_example,
}
example_titles = {
    "Generated": structure_title(
        "Generated sample", f"gen_idx={EXAMPLE_GEN_IDX}", generated_example
    ),
    "Train": structure_title(
        "Matched train sample", f"train_idx={EXAMPLE_TRAIN_IDX}", training_example
    ),
}

fig = structure_2d(
    example_structures,
    n_cols=2,
    show_cell=True,
    site_labels="legend",
    standardize_struct=False,
    subplot_title=lambda _struct, key: example_titles[key],
)
fig.update_layout(height=360, margin={"l": 10, "r": 10, "t": 80, "b": 10})
display(fig)

In [ ]:
def to_conventional_cubic_cell(structure: Structure) -> Structure:
    analyzer = SpacegroupAnalyzer(structure)
    conventional = analyzer.get_conventional_standard_structure()
    if conventional.get_space_group_info()[1] < 195:
        raise ValueError("Expected a cubic conventional cell.")
    return conventional


generated_conventional = to_conventional_cubic_cell(generated_example)
training_conventional = to_conventional_cubic_cell(training_example)

cif_dir = RESULTS_DIR / "ai4am"
cif_dir.mkdir(parents=True, exist_ok=True)
generated_cif_path = cif_dir / "mattergen_gen_8035_CoO.cif"
training_cif_path = cif_dir / "train_3438_MnSe.cif"

generated_conventional.to(filename=generated_cif_path)
training_conventional.to(filename=training_cif_path)

display(
    Markdown(f"Wrote CIF files:  \n- `{generated_cif_path}`  \n- `{training_cif_path}`")
)